# KuchoLM training

日本語コーパスを MeCab で NIDA_FICTION へ変換し、その JSONL から小型 seq2seq Transformer を Colab GPU で学習します。

この notebook だけで、データ生成 → 10件確認 → tokenizer 学習 → モデル学習 → 保存 → 推論まで通します。

In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece

## 1. 設定

In [ ]:
from pathlib import Path
import json, math, random, re
import MeCab
import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

OUTPUT_PATH = Path('/content/kucholm_nida.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
MAX_ROWS = 100_000
USE_LOCAL_TEXT = False
LOCAL_TEXT_PATH = Path('/content/corpus.txt')
DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
tagger = MeCab.Tagger()

## 2. コーパス読み込み

In [ ]:
if USE_LOCAL_TEXT:
    corpus = (line.rstrip('\n') for line in LOCAL_TEXT_PATH.open(encoding='utf-8'))
else:
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    corpus = (str(row[TEXT_COLUMN]) for row in dataset)

## 3. NIDA_FICTION 変換

In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
SENTENCE_SPLIT_RE = re.compile(r'(.+?[。！？!?]+|.+$)', re.S)

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            f = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': f[0] if len(f) > 0 else '',
                'ctype': f[4] if len(f) > 4 else '*',
                'lemma': f[7] if len(f) > 7 else '*',
                'orth_base': f[10] if len(f) > 10 else '*',
            })
        node = node.next
    return tokens

def dictionary_form(token):
    for key in ('orth_base', 'lemma'):
        value = token.get(key, '*')
        if value not in {'', '*'} and re.search(r'[ぁ-ん一-龯]', value):
            return value
    return token['surface']

def is_ichidan(token, base):
    ctype = token.get('ctype', '')
    if '下一段' in ctype or '上一段' in ctype or '一段' in ctype:
        return True
    return base.endswith('る') and token.get('surface', '') == base[:-1]

def ta_form(base, token):
    if base == '行く': return '行った'
    if base == '来る': return '来た'
    if base == 'する': return 'した'
    if is_ichidan(token, base): return base[:-1] + 'た'
    if base.endswith(('う', 'つ', 'る')): return base[:-1] + 'った'
    if base.endswith(('む', 'ぶ', 'ぬ')): return base[:-1] + 'んだ'
    if base.endswith('く'): return base[:-1] + 'いた'
    if base.endswith('ぐ'): return base[:-1] + 'いだ'
    if base.endswith('す'): return base[:-1] + 'した'
    return base + 'た'

def nai_form(base, token):
    if base == 'する': return 'しない'
    if base == '来る': return '来ない'
    if is_ichidan(token, base): return base[:-1] + 'ない'
    if base.endswith('う'): return base[:-1] + 'わない'
    mapping = {'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    last = base[-1:]
    return base[:-1] + mapping[last] + 'ない' if last in mapping else base + 'ない'

def soft_ending(body, is_question):
    if is_question: return 'ニカ'
    if re.search(r'(ね|よ|な)$', body): return 'ニダ'
    if re.search(r'(ない|ません|難しい|心配|残念|大丈夫)$', body): return 'ニダね'
    return 'ニダよ'

def split_sentences_preserve(text):
    parts = []
    cursor = 0
    for match in SENTENCE_SPLIT_RE.finditer(text):
        start, end = match.span()
        if start > cursor:
            parts.append((text[cursor:start], False))
        parts.append((match.group(0), True))
        cursor = end
    if cursor < len(text):
        parts.append((text[cursor:], False))
    return parts

def soften_surface(text):
    for pattern, replacement in [
        (r'ということです$', 'ってこと'),
        (r'ということでした$', 'ってことだった'),
        (r'のであります$', 'んだ'),
        (r'であります$', 'なんだ'),
        (r'なのです$', 'なんだ'),
        (r'のです$', 'んだ'),
        (r'でしょう$', 'だろう'),
        (r'ではありません$', 'じゃない'),
        (r'ではないです$', 'じゃない'),
        (r'ではない$', 'じゃない'),
    ]:
        text = re.sub(pattern, replacement, text)
    return text

def preserve_spaces(source, converted):
    src_spaces = [(m.start(), m.group(0)) for m in re.finditer(r' +', source)]
    if not src_spaces:
        return converted
    out = converted
    for pos, spaces in src_spaces:
        if pos < len(out) and not out[pos:pos + len(spaces)] == spaces:
            left = source[max(0, pos - 8):pos]
            right = source[pos + len(spaces):pos + len(spaces) + 8]
            if left and right:
                pattern = re.escape(left) + re.escape(right)
                replacement = left + spaces + right
                out = re.sub(pattern, replacement, out, count=1)
    return out

def auxiliary_tail(body):
    for pattern, replacement in [
        (r'てきちゃいました$', 'てきちゃった'),
        (r'て来ちゃいました$', 'て来ちゃった'),
        (r'てきました$', 'てきた'),
        (r'て来ました$', 'て来た'),
        (r'てこられました$', 'てこられた'),
        (r'て来られました$', 'て来られた'),
        (r'ていきました$', 'ていった'),
        (r'て行きました$', 'て行った'),
        (r'てしまいました$', 'てしまった'),
        (r'でしまいました$', 'でしまった'),
        (r'できました$', 'できた'),
    ]:
        if re.search(pattern, body):
            return re.sub(pattern, replacement, body)
    return None

def convert_polite_tail(body):
    direct = [
        (r'かもしれません$', 'かもしれない'),
        (r'わかりません$', 'わからない'),
        (r'知りません$', '知らない'),
        (r'いけません$', 'いけない'),
        (r'ありません$', 'ない'),
        (r'ございました$', 'あった'),
        (r'ございます$', 'ある'),
    ]
    for pattern, replacement in direct:
        if re.search(pattern, body):
            return re.sub(pattern, replacement, body)

    aux = auxiliary_tail(body)
    if aux is not None:
        return aux

    tokens = parse_tokens(body)
    if not tokens:
        return body
    surfaces = [t['surface'] for t in tokens]
    trailing_particle = ''
    if surfaces and surfaces[-1] in {'ね', 'よ', 'な'}:
        trailing_particle = surfaces.pop()
        tokens = tokens[:-1]

    patterns = [
        (['ませ', 'ん', 'でし', 'た'], 'negative_past'),
        (['ませ', 'ん'], 'negative'),
        (['まし', 'た'], 'past'),
        (['ます'], 'present'),
    ]
    for suffix, mode in patterns:
        if len(surfaces) < len(suffix) or surfaces[-len(suffix):] != suffix:
            continue
        suffix_start = len(tokens) - len(suffix)
        verb_index = next((i for i in range(suffix_start - 1, -1, -1) if tokens[i]['pos'] == '動詞'), None)
        if verb_index is None:
            continue
        verb = tokens[verb_index]
        base = dictionary_form(verb)
        prefix = ''.join(t['surface'] for t in tokens[:verb_index])

        # 受身・可能・尊敬の連鎖は最後の動詞だけを無理に五段化せず、表層語幹を保つ。
        if verb_index > 0 and any(x in ''.join(t['surface'] for t in tokens[max(0, verb_index - 2):verb_index + 1]) for x in ('られ', 'され', 'こられ', 'おられ')):
            stem = ''.join(t['surface'] for t in tokens[:suffix_start])
            if mode == 'past': return stem + 'た' + trailing_particle
            if mode == 'present': return stem + trailing_particle

        if mode == 'present': replacement = base
        elif mode == 'past': replacement = ta_form(base, verb)
        else:
            negative = nai_form(base, verb)
            replacement = negative if mode == 'negative' else negative[:-2] + 'なかった'
        return prefix + replacement + trailing_particle

    if surfaces[-2:] == ['でし', 'た']:
        return ''.join(surfaces[:-2]) + 'だった' + trailing_particle
    if surfaces[-1:] == ['です']:
        return ''.join(surfaces[:-1]) + trailing_particle
    return body

def convert_sentence(sentence):
    m = re.match(r'^(\s*)(.*?)(\s*)$', sentence, re.S)
    leading, core, trailing = m.groups()
    if not core or URL_RE.search(core):
        return sentence
    punct_match = re.search(r'([。！？!?]+)$', core)
    punctuation = punct_match.group(1) if punct_match else ''
    body = core[:-len(punctuation)] if punctuation else core
    is_question = bool(re.search(r'[？?]$', punctuation))
    softened = soften_surface(body)
    if softened.endswith('か') and is_question:
        softened = softened[:-1]
    converted = convert_polite_tail(softened)
    if converted.endswith(('ね', 'よ', 'な')):
        particle = converted[-1]
        converted = converted[:-1] + 'ニダ' + particle
    else:
        converted += soft_ending(softened, is_question)
    converted = preserve_spaces(body, converted)
    return leading + converted + (punctuation or ('？' if is_question else '')) + trailing

def to_nida(text):
    if not text or URL_RE.search(text):
        return None
    return ''.join(convert_sentence(part) if is_sentence else part for part, is_sentence in split_sentences_preserve(text))


## 4. JSONL生成

In [ ]:
written = 0
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for source in corpus:
        if not source or len(source.strip()) < 2 or len(source) > 256:
            continue
        target = to_nida(source)
        if not target or target == source:
            continue
        output.write(json.dumps({'style':'NIDA_FICTION','source':source,'target':target}, ensure_ascii=False) + '\n')
        written += 1
        if written >= MAX_ROWS:
            break
print('written:', written)
print('size MB:', OUTPUT_PATH.stat().st_size / 1024 / 1024)

## 5. 生成サンプルを10件確認

In [ ]:
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        item = json.loads(line)
        print(f"{i + 1:02d}. {item['source']} -> {item['target']}")

## 6. 品質チェック

In [ ]:
checks = {'space_lost': [], 'polite_left': [], 'broken_aux': []}
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        s, t = item['source'], item['target']
        if s.count(' ') > t.count(' '): checks['space_lost'].append((s, t))
        if re.search(r'(ました|ません)(?:ニダ|ニカ)', t): checks['polite_left'].append((s, t))
        if re.search(r'(くった|ござった|おくった|こられった)', t): checks['broken_aux'].append((s, t))
print({k: len(v) for k, v in checks.items()})
for name, rows in checks.items():
    print('\n', name)
    for s, t in rows[:5]:
        print(s, '\n ->', t, '\n')